### Binary Traffic Sign Classification

This is our first notebook on neural networks. We will start by building a simple neural network that will help us to classify two types of traffic signs. The signs used in this lesson are a processed subset (28x28 grayscale images) from the famous [German Traffic Sign Recognition Benchmark (GTSRB)](https://benchmark.ini.rub.de/gtsrb_dataset.html).

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

import pickle
from time import time
import numpy as np
from sklearn.utils import shuffle

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 6]

#### Data Loading

Let's now load the data to see what we are dealing with. The dataset is stored as a dictionary with one list of images for each class.

In [ ]:
with open('data/subset_lecture.pckl', 'rb') as f:
    data = pickle.load(f)

images_0 = data['class_id_0']
images_1 = data['class_id_1']
labels_0 = [0] * len(images_0)
labels_1 = [1] * len(images_1)

print('Image shape', images_0[0].shape, images_0[0].dtype)

Let's have a look at some random samples of both classes. The first row shows class 0 (the *yield* sign), the second row class 1 (the *priority road* sign). Notice how different the lighting conditions are, some of the signs are almost black, others are almost white.

In [ ]:
for row, (images, labels) in enumerate([(images_0, labels_0), (images_1, labels_1)]):
    for cnt, idx in enumerate(np.random.randint(0, len(images), 10)):
        plt.subplot(2, 10, row*10 + cnt + 1)
        plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
        plt.title(labels[idx]), plt.axis(False)

How many samples do we have for each class? It is always a good idea to check this. If one class were much more frequent than the other, a classifier that always predicts the frequent class would already look good in terms of accuracy.

In [ ]:
print('Num samples class_0', len(images_0))
print('Num samples class_1', len(images_1))
print('Accuracy of always predicting class 0:', len(images_0) / (len(images_0) + len(images_1)))

#### Prepare Input Data

To train our neural network model, we have to prepare the data in the format the model actually expects. In our case, this will be numpy arrays. We put both classes together and **shuffle** them, so that training batches contain both classes (images and labels are shuffled together, so they stay matched).

In [ ]:
images = images_0 + images_1
labels = labels_0 + labels_1
images, labels = shuffle(images, labels)

images = np.array(images)
labels = np.array(labels)

print('Images', images.shape)
print('Labels', labels.shape)

But now we have a problem. We cannot just feed an image to a neuron since the neuron inputs are flat (one dimensional). On the other hand, the images are 2D matrices. Therefore, we need to "flatten" each image to a one dimensional vector of pixels (28x28 = 784 values). We will also scale the pixel values from 0-255 to 0-1.

The first approach is the most explicit one, we simply loop over all the pixels:

In [ ]:
start = time()
pixels = []
for image in images:
    pixels_ = []
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            pixels_.append(image[r,c])
    pixels.append(pixels_)

pixels = np.array(pixels)/255
stop = time()

print('Shape', pixels.shape)
print('Elapsed time', stop - start)

Or, you know, just let's make use of our friend numpy :-)

In [ ]:
start = time()
pixels = np.array([image.flatten() for image in images])/255
stop = time()

print('Shape', pixels.shape)
print('Elapsed time', stop - start)

Before the training, let's again have a look at some random samples. This time, we also print the labels, so that we can check that the shuffling did not mix up the images and their labels.

In [ ]:
for cnt, idx in enumerate(np.random.randint(0, len(images), 10)):
    plt.subplot(2,5,cnt+1)
    plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title(labels[idx]), plt.axis(False)

#### Building the Neural Network

Let's now build our first (and yes, very simple) neural network using Keras. For that, we will need a couple of imports.

In [ ]:
import keras
from keras import Model
from keras.layers import Dense, Input

The following network will consist of only one single neuron. It is a very tiny network (not even a network, strictly speaking :-) ) and yet it can be quite powerful. The neuron has one weight for each pixel (784) and one bias, which makes 785 trainable parameters.

In [ ]:
inputs = Input(shape=(pixels.shape[1],))
outputs = Dense(1, activation="linear")(inputs)
model = Model(inputs, outputs)

model.summary()

# Remember the (random) initial weights, we will need them later
weights_init = model.layers[1].get_weights()[0].copy()    # copy, since the training changes the weights

In [ ]:
keras.utils.plot_model(model, to_file="model.png", show_shapes=True, show_layer_names=True)

Before starting the training, we have to compile the model. During the compilation, we indicate what optimizer we want to use and what loss should be applied for the minimization process. Here we use the Adam optimizer (one of the variants of stochastic gradient descent) and the mean squared error between the neuron output and the label (0 or 1).

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')

And let's train :-) The data are processed in mini-batches of 32 images and one pass through the whole dataset is one epoch.

In [ ]:
history = model.fit(pixels, labels, epochs=10, batch_size=32, verbose=2)

In [ ]:
# Plot training history
h = history.history
epochs = range(len(h['loss']))
plt.plot(epochs, h['loss'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.ylabel('loss');

#### Learnt Weights

Let's also have a look at what the neuron has learnt. It has one weight per pixel, so we can reshape the 784 weights back to a 28x28 image. We show the initial (random) weights, the weights after the training and their difference, i.e. what the training has changed.

In [ ]:
weights, bias = model.layers[1].get_weights()
print('Bias', bias)

for cnt, (title, w) in enumerate([('Initial weights', weights_init),
                                  ('Trained weights', weights),
                                  ('Trained - initial', weights - weights_init)]):
    plt.subplot(1, 3, cnt+1)
    plt.imshow(w.reshape(28, 28), cmap='gray'), plt.colorbar(shrink=0.7)
    plt.title(title), plt.axis(False)

The trained weights still look quite random, because they contain the random initialization and ten epochs are not enough to get rid of it. The difference to the initial weights is much more informative. It shows that the neuron has mostly changed the weights in the center of the image, where it now has **negative** weights. This makes sense, since the signs of class 0 (yield) are on average brighter in the center than the signs of class 1, so a bright center pushes the output of the neuron towards 0.

#### Performance Evaluation

Once our model is trained, we can run it on our images to see how it performs (inference). Let's start with a single image:

In [ ]:
idx = 50
pred = model.predict(pixels[idx:idx+1, ...], verbose=0)
print(pred, labels[idx])

The output of the neuron is a real number and not exactly 0 or 1. To get a class, we threshold it at 0.5. Let's run the model on the entire dataset and compute the accuracy.

In [ ]:
# Run it on the entire dataset
predictions = model.predict(pixels, verbose=0).squeeze()
predictions = (predictions > 0.5).astype(int)

In [ ]:
correct = 0
for prediction, label in zip(predictions, labels):
    if prediction == label:
        correct = correct + 1

print('Accuracy', correct/len(labels))

Keep in mind that this is the accuracy on the **same** images the model was trained on, so it is optimistic. How the model behaves on unseen data is a topic for one of the next lessons.

Let's look at some predictions...

In [ ]:
for cnt, idx in enumerate(np.random.randint(0, len(images), 10)):
    plt.subplot(2,5,cnt+1), plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title('Label: ' + str(labels[idx]) + ' | Prediction: ' + str(predictions[idx]))
    plt.axis(False)

... and, more interestingly, at the mistakes. Many of them are very dark or low contrast images, where the sign is hard to recognize even for us.

In [ ]:
wrong = np.where(predictions != labels)[0]
print('Number of mistakes', len(wrong))

for cnt, idx in enumerate(wrong[:10]):
    plt.subplot(2,5,cnt+1), plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title('Label: ' + str(labels[idx]) + ' | Prediction: ' + str(predictions[idx]))
    plt.axis(False)